In [9]:
import pandas as pd
import numpy as np
import sys
sys.path.append('../kaggle_prediction_library/') 
import preprocess
import feature_engineering
import submission
from sklearn.model_selection import train_test_split

# from hyperopt import tpe, fmin, Trials
# import hyperopt.hp as hp


In [10]:
from sklearn.linear_model import LogisticRegression
from sklearn.linear_model import LogisticRegressionCV
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report
from sklearn.metrics import log_loss
from sklearn.feature_selection import chi2
from sklearn.metrics import r2_score
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import brier_score_loss


### Read data

In [11]:
regular_season_results = pd.read_csv('../data/MRegularSeasonDetailedResults.csv')
detailed_tourney_results = pd.read_csv('../data/MNCAATourneyDetailedResults.csv')
rankings = pd.read_csv('../data/MMasseyOrdinals.csv')
seeds = pd.read_csv('../data/MNCAATourneySeeds.csv')

# kp_rankings = pd.read_csv('../data/kenpom_pre_tourney_snapshot.csv')

regular_season_results_w = pd.read_csv('../data/WRegularSeasonDetailedResults.csv')
detailed_tourney_results_w = pd.read_csv('../data/WNCAATourneyDetailedResults.csv')

mteams = pd.read_csv('../data/MTeams.csv')
wteams = pd.read_csv('../data/WTeams.csv')

seeds_w = pd.read_csv('../data/WNCAATourneySeeds.csv')

# M538 = pd.read_csv('../data/M538.csv')
# W538 = pd.read_csv('../data/W538.csv')

seed_round = pd.read_csv("../data/MNCAATourneySeedRoundSlots.csv")
seeds = pd.read_csv("../data/MNCAATourneySeeds.csv")

first_round_odds_data = pd.read_csv('../data/sky_data/first_round_odds_ncaam.csv')
first_round_odds_data_w = pd.read_csv('../data/sky_data/first_round_odds_ncaaw.csv')


In [12]:
torvik_player_data = pd.read_csv("../data/sky_data/torvik_player_data_2008_2024.csv")
torvik_player_data_ncaaw = pd.read_csv("../data/sky_data/ncaaw_torvik_player_data_2021_2024.csv")


In [13]:
mapping = pd.read_csv("../data/sky_data/mappings/kaggle_torvik_mapping.csv")
mapping[mapping.Kaggle_Team.str.contains("Sa")]

,Kaggle_Team,Final_Torvik_Team
85,San Diego,San Diego
103,UT San Antonio,UTSA
105,San Diego St,San Diego St.
116,La Salle,La Salle
175,Sam Houston St,Sam Houston St.
219,UC Santa Barbara,UC Santa Barbara
264,Samford,Samford
271,Sacred Heart,Sacred Heart
291,San Francisco,San Francisco
308,CS Sacramento,Sacramento St.


In [14]:
sorted(torvik_player_data.Team.unique())

['Abilene Christian',
 'Air Force',
 'Akron',
 'Alabama',
 'Alabama A&M',
 'Alabama St.',
 'Albany',
 'American',
 'Appalachian St.',
 'Arizona',
 'Arizona St.',
 'Arkansas',
 'Arkansas Pine Bluff',
 'Auburn',
 'Austin Peay',
 'BYU',
 'Baylor',
 'Belmont',
 'Binghamton',
 'Boise St.',
 'Boston College',
 'Boston University',
 'Bradley',
 'Bryant',
 'Bucknell',
 'Buffalo',
 'Butler',
 'Cal Poly',
 'Cal St. Bakersfield',
 'Cal St. Fullerton',
 'Cal St. Northridge',
 'California',
 'Central Connecticut',
 'Central Michigan',
 'Charleston',
 'Charlotte',
 'Chattanooga',
 'Cincinnati',
 'Clemson',
 'Cleveland St.',
 'Coastal Carolina',
 'Colgate',
 'Colorado',
 'Colorado St.',
 'Connecticut',
 'Coppin St.',
 'Cornell',
 'Creighton',
 'Davidson',
 'Dayton',
 'DePaul',
 'Delaware',
 'Delaware St.',
 'Detroit Mercy',
 'Drake',
 'Drexel',
 'Duke',
 'Duquesne',
 'East Tennessee St.',
 'Eastern Kentucky',
 'Eastern Washington',
 'Fairleigh Dickinson',
 'Florida',
 'Florida A&M',
 'Florida Atlanti

In [15]:
#aggregated_player_stats = pd.read_csv("../data/sky_data/aggregated_player_stats.csv")

In [16]:
sub_df = pd.read_csv("SampleSubmission2024.csv")

### Set Up Data

In [17]:
to_predict_mens, to_predict_womens, regular_season_games, regular_season_games_w = preprocess.full_setup(detailed_tourney_results, regular_season_results,
               detailed_tourney_results_w, regular_season_results_w,
               sub_df, mteams)

/Users/skylerdale/workspace/kaggle-ncaam/2025_model/production_notebooks/../kaggle_prediction_library/submission.py:45: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  return pd.concat([mens_historical_games, womens_historical_games, sub], axis = 0)


### Add Features

In [18]:
to_predict_womens = feature_engineering.TournamentSeed(tourney_seeds=seeds_w).add(to_predict_womens)
to_predict_womens = feature_engineering.Efficiency(games=regular_season_games_w, away_bonus=0).add(to_predict_womens)
to_predict_womens = feature_engineering.RoundNumber(seeds, seed_round).add(to_predict_womens)
to_predict_womens = feature_engineering.TeamNames(wteams).add(to_predict_womens)

# to_predict_womens = feature_engineering.FiveThirtyEight(fivethirtyeight_df=W538).add(to_predict_womens)

/Users/skylerdale/workspace/kaggle-ncaam/2025_model/production_notebooks/../kaggle_prediction_library/feature_engineering.py:132: FutureWarning: The provided callable <function mean at 0x1063bc0d0> is currently using SeriesGroupBy.mean. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "mean" instead.
  final = all_games3.groupby(['Season', 'Team1']).agg(adj_oe=('adj_oe', np.mean), adj_de=('adj_de', np.mean)).reset_index()


In [19]:
to_predict_womens = feature_engineering.FirstRoundOdds(first_round_odds_data_w).add(to_predict_womens)

In [20]:
# to remove later
to_predict_womens = to_predict_womens[(to_predict_womens.type != "Prediction")].copy()

In [21]:
to_predict_womens = feature_engineering.RoundNumber(seeds, seed_round).add(to_predict_womens)
to_predict_womens = feature_engineering.AggregatedPlayerStats(torvik_player_data_ncaaw).add(to_predict_womens)

In [22]:
#to_predict_womens.to_csv("../development_notebooks/to_predict_women.csv")
to_predict_womens.to_csv("to_predict_women.csv")

In [23]:
to_predict_mens = feature_engineering.TeamNames(mteams).add(to_predict_mens)
to_predict_mens = feature_engineering.FirstRoundOdds(first_round_odds_data).add(to_predict_mens)
to_predict_mens = feature_engineering.RoundNumber(seeds, seed_round).add(to_predict_mens)
to_predict_mens = feature_engineering.SeasonStats(regular_season_games).add(to_predict_mens)
# to_predict_mens = feature_engineering.FiveThirtyEight(fivethirtyeight_df=M538).add(to_predict_mens)
to_predict_mens = feature_engineering.PreSeasonAPRankings(rankings_df=rankings).add(to_predict_mens)
to_predict_mens = feature_engineering.TournamentSeed(tourney_seeds=seeds).add(to_predict_mens)
to_predict_mens = feature_engineering.Efficiency(games=regular_season_games, away_bonus=0).add(to_predict_mens)
to_predict_mens = feature_engineering.FinalRanking(rankings_df=rankings, system='WLK').add(to_predict_mens) # switched in 2024 because SAG dissapeared
# to_predict_mens = feature_engineering.Kenpom(kp_snapshot=kp_rankings).add(to_predict_mens)
# this one takes 3 minutes to run
# to_predict_mens = feature_engineering.TeamQuality(games=regular_season_games).add(to_predict_mens)


/Users/skylerdale/workspace/kaggle-ncaam/2025_model/production_notebooks/../kaggle_prediction_library/feature_engineering.py:223: FutureWarning: The provided callable <function mean at 0x1063bc0d0> is currently using DataFrameGroupBy.mean. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "mean" instead.
  season_statistics = df.groupby(["Season", 'Team1'])[boxscore_cols].agg(np.mean).reset_index()
/Users/skylerdale/workspace/kaggle-ncaam/2025_model/production_notebooks/../kaggle_prediction_library/feature_engineering.py:132: FutureWarning: The provided callable <function mean at 0x1063bc0d0> is currently using SeriesGroupBy.mean. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "mean" instead.
  final = all_games3.groupby(['Season', 'Team1']).agg(adj_oe=('adj_oe', np.mean), adj_de=('adj_de', np.mean)).reset_index()
/Users/skylerdale/workspace/kaggle-nc

In [24]:
to_predict_mens = feature_engineering.AggregatedPlayerStats(torvik_player_data).add(to_predict_mens)


In [25]:
# Original 

# to_predict_mens = to_predict_mens[(to_predict_mens.type != "Prediction") & 
#                                   (to_predict_mens.final_odds.notnull())
#                                   ]

# New - Not Prediction and (game round != 1 OR final odds is null)

# to_predict_mens = to_predict_mens[
#                                   ( (to_predict_mens.final_odds.notnull()) | (to_predict_mens.GameRound != 1) )
                               
#                                   ]

### Split Dataset

In [26]:
# first_round_df = to_predict_mens[to_predict_mens.GameRound == 1].copy()
# other_rounds_df = to_predict_mens[to_predict_mens.GameRound > 1].copy()

In [27]:
# first_round_df.to_csv("to_predict_mens_first_round.csv")
# other_rounds_df.to_csv("to_predict_mens_other_rounds.csv")
to_predict_mens.to_csv("to_predict_mens.csv")